<a href="https://colab.research.google.com/github/jaceksta/nlp/blob/main/NLP_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [144]:
!pip install pandas matplotlib nltk wordcloud scikit-learn morfeusz2

In [145]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [146]:
corpus_dir = '/content/Literature'

In [147]:
from nltk.corpus import PlaintextCorpusReader
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np
import operator as op
import itertools as it
import pandas as pd
import morfeusz2
import json
import string

In [148]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [149]:
def print_dictionary(dictionary):
  print(json.dumps(dictionary, indent = 4, ensure_ascii=False))


In [150]:
corpus = PlaintextCorpusReader(corpus_dir, r".*\.txt")

In [151]:
corpus

<PlaintextCorpusReader in '/content/Literature'>

In [152]:
files_names = corpus.fileids()

In [153]:
files_names

['Harry Potter i Czara Ognia.txt',
 'Harry Potter i Insygnia Smierci.txt',
 'Harry Potter i Kamien Filozoficzny.txt',
 'Harry Potter i Komnata Tajemnic.txt',
 'Harry Potter i Ksiaze Polkrwi.txt',
 'Harry Potter i Przeklete Dziecko.txt',
 'Harry Potter i Wiezien Azkabanu.txt',
 'Harry Potter i Zakon Feniksa.txt',
 'Opowiesci z Narni. Kon i jego chlopiec.txt',
 'Opowiesci z Narni. Ksiaze Kaspian.txt',
 'Opowiesci z Narni. Lew, czarownica i stara szafa.txt',
 'Opowiesci z Narni. Ostatnia bitwa.txt',
 'Opowiesci z Narni. Podroz Wedrowca do Switu.txt',
 'Opowiesci z Narni. Siostrzeniec Czarodzieja.txt',
 'Opowiesci z Narni. Srebrne krzeslo.txt',
 'Zmierzch.Ksiezyc w Nowiu.txt',
 'Zmierzch.Przed Switem.txt',
 'Zmierzch.Zacmienie.txt',
 'Zmierzch.Zmierzch.txt',
 'Zmierzch.Zycie i Smierc.txt']

#Preprocessing

In [154]:
from codecs import encode
documents_dict = {}

for file_name in files_names:
  documents_dict[file_name] = corpus.raw(file_name)

#print(json.dumps(documents_dict, indent = 4, ensure_ascii=False))



In [155]:
documents_lengths = {}

for file_name in files_names:
  documents_lengths[file_name] = {
      "pre": len(word_tokenize(documents_dict[file_name]))
  }

#print(json.dumps(documents_lengths, indent = 4, ensure_ascii=False))


In [156]:
documents_df = pd.DataFrame.from_dict(documents_dict, orient="index")
documents_df.columns = ['content']

documents_df

,content
Harry Potter i Czara Ognia.txt,"Lord Voldemort, wraz ze swoim sługą Glizdogone..."
Harry Potter i Insygnia Smierci.txt,"Podczas zebrania śmierciożerców, które odbywa ..."
Harry Potter i Kamien Filozoficzny.txt,Na ulicy Privet Drive pojawia się stary człowi...
Harry Potter i Komnata Tajemnic.txt,"Wuj Vernon jest zły, że Hedwiga robi bałagan w..."
Harry Potter i Ksiaze Polkrwi.txt,Mugolskiego premiera odwiedza Korneliusz Knot ...
Harry Potter i Przeklete Dziecko.txt,"Scena pierwsza oraz druga opowiada historię, k..."
Harry Potter i Wiezien Azkabanu.txt,Harry leży pod kocami w łóżku i odrabia prace ...
Harry Potter i Zakon Feniksa.txt,Dudley jest zły i wyżywa się na Harrym. Dudley...
Opowiesci z Narni. Kon i jego chlopiec.txt,Akcja książki rozgrywa się w Narnii za czasów ...
Opowiesci z Narni. Ksiaze Kaspian.txt,"Mija rok od wydarzeń opisanych w powieści ""Lew..."


In [157]:
matrix_tf = CountVectorizer().fit_transform(documents_df['content'])

matrix_tf.shape

(20, 7469)

In [158]:
sparsity_tf = matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1])
sparsity_tf

0.09044048734770384

In [159]:
matrix_parameters = {
    "pre": {
      'words': matrix_tf.shape[1],
      "sparsity": sparsity_tf
    }
}

print_dictionary(matrix_parameters)

{
    "pre": {
        "words": 7469,
        "sparsity": 0.09044048734770384
    }
}


In [160]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding = "utf-8")

stopwords = stopwords_file.read().splitlines()

stopwords_file.close()
stopwords = stopwords[4:]
#stopwords

In [161]:
morfer = morfeusz2.Morfeusz()

print(*morfer.analyse("Ala ma kota."), sep='\n')

(0, 1, ('Ala', 'Ala', 'subst:sg:nom:f', ['imię'], []))
(0, 1, ('Ala', 'Al', 'subst:sg:gen.acc:m1', ['imię'], []))
(0, 1, ('Ala', 'Alo', 'subst:sg:gen.acc:m1', ['imię'], []))
(1, 2, ('ma', 'mieć', 'fin:sg:ter:imperf', [], []))
(1, 2, ('ma', 'mój:A', 'adj:sg:nom.voc:f:pos', [], []))
(2, 3, ('kota', 'kota', 'subst:sg:nom:f', ['nazwa_pospolita'], []))
(2, 3, ('kota', 'kot:Sm1', 'subst:sg:gen.acc:m1', ['nazwa_pospolita'], ['pot.,środ.']))
(2, 3, ('kota', 'kot:Sm2', 'subst:sg:gen.acc:m2', ['nazwa_pospolita'], []))
(3, 4, ('.', '.', 'interp', [], []))


In [162]:
def lemmatize(text):
  morfer = morfeusz2.Morfeusz()
  segments = it.groupby(morfer.analyse(text), op.itemgetter(0,1))
  def disambiguate(group):
    pairs = ((len(descr), lemma) for _, _, (_, lemma, descr, _, _, ) in group)
    _, lemma = min(pairs)
    return lemma.split(":")[0]
  return " ".join(filter(str.isalpha, (disambiguate(group) for _, group in segments)))


In [163]:
for file_name in documents_dict:
  documents_dict[file_name] = documents_dict[file_name].lower()
  documents_dict[file_name] = "".join(filter(lambda char: char not in string.punctuation, documents_dict[file_name]))
  documents_dict[file_name] = lemmatize(documents_dict[file_name])
  documents_dict[file_name] = " ".join(filter(lambda word: word not in stopwords, word_tokenize(documents_dict[file_name], language='polish')))



In [164]:


#print(json.dumps(documents_lengths, indent = 4, ensure_ascii=False))


In [165]:
documents_df = pd.DataFrame.from_dict(documents_dict, orient="index")
documents_df.columns = ['content']
matrix_tf = CountVectorizer().fit_transform(documents_df['content'])
sparsity_tf = matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1])

matrix_parameters['post'] = {
      'words': matrix_tf.shape[1],
      "sparsity": sparsity_tf
}




print_dictionary(matrix_parameters)

{
    "pre": {
        "words": 7469,
        "sparsity": 0.09044048734770384
    },
    "post": {
        "words": 4675,
        "sparsity": 0.10687700534759358
    }
}


In [166]:
for file_name in files_names:
  documents_lengths[file_name]['post'] = len(word_tokenize(documents_dict[file_name], language='polish'))

In [167]:
documents_lengths = pd.DataFrame.from_dict(documents_lengths, orient = 'index')
documents_lengths.columns = ['pre', 'post']
display(documents_lengths)
documents_lengths['diff'] = documents_lengths['post'] - documents_lengths['pre']
documents_lengths['perc_diff'] = documents_lengths['diff'] / documents_lengths['pre']
display(documents_lengths)


,pre,post
Harry Potter i Czara Ognia.txt,6772,3806
Harry Potter i Insygnia Smierci.txt,3035,1681
Harry Potter i Kamien Filozoficzny.txt,2244,1234
Harry Potter i Komnata Tajemnic.txt,1636,888
Harry Potter i Ksiaze Polkrwi.txt,2039,1141
Harry Potter i Przeklete Dziecko.txt,3178,1742
Harry Potter i Wiezien Azkabanu.txt,2233,1241
Harry Potter i Zakon Feniksa.txt,1835,1039
Opowiesci z Narni. Kon i jego chlopiec.txt,179,106
Opowiesci z Narni. Ksiaze Kaspian.txt,2376,1299


,pre,post,diff,perc_diff
Harry Potter i Czara Ognia.txt,6772,3806,-2966,-0.437980
Harry Potter i Insygnia Smierci.txt,3035,1681,-1354,-0.446129
Harry Potter i Kamien Filozoficzny.txt,2244,1234,-1010,-0.450089
Harry Potter i Komnata Tajemnic.txt,1636,888,-748,-0.457213
Harry Potter i Ksiaze Polkrwi.txt,2039,1141,-898,-0.440412
Harry Potter i Przeklete Dziecko.txt,3178,1742,-1436,-0.451857
Harry Potter i Wiezien Azkabanu.txt,2233,1241,-992,-0.444245
Harry Potter i Zakon Feniksa.txt,1835,1039,-796,-0.433787
Opowiesci z Narni. Kon i jego chlopiec.txt,179,106,-73,-0.407821
Opowiesci z Narni. Ksiaze Kaspian.txt,2376,1299,-1077,-0.453283


In [169]:
matrix_parameters = pd.DataFrame.from_dict(matrix_parameters, orient='index')
matrix_parameters.columns = ['words', 'sparsity']
display(matrix_parameters)

,words,sparsity
pre,7469,0.090440
post,4675,0.106877
